# How MCP tool definitions change: a first look at the HISTOR snapshot

HISTOR reads the tool list of every remote MCP server in the official registry once a day and runs
new npm/PyPI MCP package versions in a sandbox; every change is recorded field by field in a signed
log. This notebook loads the monthly snapshot and reproduces the headline numbers.

Source and documentation: https://histor.modelmarket.dev/datasets/ · DOI 10.5281/zenodo.23512759 · CC BY 4.0

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# On Kaggle the dataset is mounted under /kaggle/input; elsewhere set HISTOR_DATA to a snapshot folder.
# Kaggle extracts .gz uploads; support both the mounted JSONL and the original archive.
if os.environ.get("HISTOR_DATA"):
    DATA = Path(os.environ["HISTOR_DATA"])
else:
    candidates = sorted(Path("/kaggle/input").rglob("servers.jsonl*"))
    candidates = [p for p in candidates if p.name in ("servers.jsonl", "servers.jsonl.gz")]
    if len(candidates) != 1:
        raise RuntimeError("Attach the HISTOR dataset, or set HISTOR_DATA to its snapshot folder.")
    DATA = candidates[0].parent

def snapshot_file(name):
    for suffix in (".jsonl", ".jsonl.gz"):
        path = DATA / (name + suffix)
        if path.is_file():
            return path
    raise FileNotFoundError(f"Missing {name}.jsonl or {name}.jsonl.gz in {DATA}")

servers = pd.read_json(snapshot_file("servers"), lines=True)
changes = pd.read_json(snapshot_file("changes"), lines=True)
print(f"{len(servers):,} listed servers and packages, {len(changes):,} recorded changes")

## Remote servers: how many served a different tool list within four weeks?
Remote servers that had a pinned tool list by 24 September and are still listed, and how many of them
show at least one fully read change by the snapshot date (20 October 2026).

In [ ]:
summary = changes["summaryJson"].map(json.loads)
changes["detail"] = summary.map(lambda s: s.get("detail"))
full_changers = set(changes.loc[changes["detail"] == "full", "serverId"])
base = servers[(servers["transport"] != "stdio") & (~servers["delisted"].fillna(False).astype(bool))
               & servers["toolSetDigest"].notna() & (servers["firstPinned"] <= "2026-09-24T23:59:59Z")]
changed = base["id"].isin(full_changers)
print(f"{len(base):,} remote servers, {changed.sum():,} changed ({changed.mean():.1%})")

## What kind of change?

In [ ]:
def kind(s: dict) -> str:
    if s.get("detail") != "full":
        return "only tool names readable"
    add_rm = bool(s.get("added") or s.get("removed"))
    mod = bool(s.get("modified"))
    return "both" if add_rm and mod else "tools added or removed only" if add_rm else "existing tools modified only"

kinds = summary.map(kind).value_counts()
print(pd.DataFrame({"changes": kinds, "share": (kinds / kinds.sum()).round(3)}))

In [ ]:
fields = pd.Series([field for s in summary for tool in s.get("modified", []) for field in tool.get("fields", {})])
print("field changes, counting each modified tool once per change:")
print(fields.value_counts().head(8))

## How often does one server change?

In [ ]:
per_server = changes.groupby("serverId").size()
print(per_server.describe(percentiles=[0.5, 0.9, 0.99]).round(1))
ax = per_server.clip(upper=20).plot.hist(bins=20, figsize=(8, 4), title="Changes per server (20+ grouped)")
ax.set_xlabel("recorded changes"); ax.set_ylabel("servers")
plt.tight_layout(); plt.show()

## Changes per day

In [ ]:
per_day = pd.to_datetime(changes["observedAt"]).dt.floor("D").value_counts().sort_index()
ax = per_day.plot(figsize=(8, 4), title="Recorded changes per day")
ax.set_ylabel("changes"); plt.tight_layout(); plt.show()

Pattern matches in `scans.jsonl.gz` are leads for a human to read, not verdicts, and LLM readings in
`classifications.jsonl.gz` are advisory. A change is not an attack. Check one server without
downloading anything: https://modelmarket.dev/tools/histor-check